# Galaxy Counts (LSS Observing Strategy) - MAF contrib implementation and demo


- Author: Sylvie Dagoret-Campagne
- Creation date: 2026-09-15
- Context: SCOC (Survey Cadence Optimization Committee). Unlike notebooks 01-05 (official `science_radar_batch` groups), this notebook, **07**, covers two galaxy-count utilities from `rubin_sim.maf.maf_contrib.lss_obs_strategy` - the original LSST "LSS observing strategy" mafContrib package (Awan, H. et al. 2016). They are not part of the official batch but are the standard MAF tools for propagating coadded depth into an LSS galaxy sample, and are the basis of the `depthLimitedNumGalMetric` used in Lochner et al. 2018 to estimate galaxy counts for the DESC static-probes forecasts referenced in notebook 01.
- Simulation analyzed: `/Users/dagoret/DATA/OpSim/sim_baseline/baseline_v5.3.6_10yrs.db`
- Source modules covered:
  - `galaxy_counts_metric_extended.py` -> `GalaxyCountsMetricExtended`, a real `rubin_sim.maf.metrics.BaseMetric` that can be dropped into a `MetricBundle` + `HealpixSlicer` like any other MAF metric.
  - `galaxy_counts_with_pixel_calibration.py` -> `galaxy_counts_with_pixel_calibration`, a standalone function (not a `BaseMetric` subclass) that computes the same physics for one HEALpix pixel at a time, given its coadded depth directly - i.e. it bypasses the slicer entirely. This is meant for injecting **pixel-specific calibration systematics** that are not columns in the OpSim database (e.g. a photometric zero-point error map), which a normal MAF metric run through a slicer has no way to see.


## Notebook overview

**Physical model (shared by both).** Both utilities implement the same galaxy-count model, following Awan et al. (2016):

1. Start from the coadded, extinction-corrected 5-sigma point-source depth in a pixel, `coaddm5` (from `ExgalM5`, dust-corrected) or the plain coadded depth (`Coaddm5Metric`, no dust correction).
2. Take a galaxy luminosity-function power law `dN/dm = 10**(a*m + b)` (per sq. degree, per magnitude), with coefficients `power_law_const_a` / `power_law_const_b` tabulated **per redshift bin** from mock catalogs built on the semi-analytic SAG model (Padilla et al.), imported from `constants_for_pipeline.py`. A per-band color correction shifts the power law from its native `i`-band calibration to any of `ugrizy`.
3. Multiply by an incompleteness (detection-probability) factor `0.5 * erfc(m - coaddm5)` - a smooth 50%-at-`coaddm5` completeness curve - and integrate over apparent magnitude up to `upper_mag_limit`.
4. Optionally normalize the raw mock-catalog counts to match the empirical CFHTLS `i<25.5` power law from the LSST Science Book (eq. 3.7) via `normalization_constant`, so absolute counts are anchored to real data rather than only the mock catalog's arbitrary normalization.
5. Scale the per-sq-degree count down to **one HEALpix pixel** (`41253 deg^2 / npix`).

**`GalaxyCountsMetricExtended` (per-pixel MAF metric).** Runs inside the normal MAF slicer machinery: for a given band (`filter_col` selects the matching visits from a `data_slice` that may contain several bands), it calls its internal `ExgalM5` (or `Coaddm5Metric`) parent metric to get `coaddm5` for that pixel, then integrates the power law as above and returns the galaxy count for that pixel. Options: `redshift_bin` (one z-slice or `'all'`), `cfht_ls_counts` (CFHTLS power law instead of the mock-catalog sum), `normalized_mock_catalog_counts`, `include_dust_extinction`, `upper_mag_limit`, `nside` (only used for the sq-deg -> pixel scale factor, so must match the slicer's own `nside`).

**`galaxy_counts_with_pixel_calibration` (standalone pseudo-metric).** Exactly the same physics, but as a plain function taking `coaddm5` directly as an argument instead of a `data_slice` - "like a pseudo-metric", per its own module docstring, because it does not communicate with a slicer at all. This makes it the right tool when the `coaddm5` value you want to integrate over is not simply `ExgalM5` evaluated on OpSim visits, but has been perturbed pixel-by-pixel by something external to the cadence itself - e.g. a **pixel-dependent photometric calibration error map** - which is exactly the "necessary when accounting for pixel-specific calibration errors" use case named in the module's header comment. Section 8 below demonstrates this: a baseline `coaddm5` HEALpix map is computed once with the normal MAF chain, then `galaxy_counts_with_pixel_calibration` is called pixel-by-pixel, once on the unperturbed map and once after adding a toy per-pixel calibration-offset map, to see how a calibration systematic propagates into galaxy-count fluctuations.


## Simulation and code references
- OpSim run analyzed: `baseline_v5.3.6_10yrs.db` (Rubin baseline v5.3.6, 10-year simulation)
- `rubin_sim.maf` source (main branch, retrieved for this notebook):
  - `rubin_sim/maf/maf_contrib/lss_obs_strategy/galaxy_counts_metric_extended.py` (`GalaxyCountsMetricExtended`) - https://github.com/lsst/rubin_sim/blob/main/rubin_sim/maf/maf_contrib/lss_obs_strategy/galaxy_counts_metric_extended.py
  - `rubin_sim/maf/maf_contrib/lss_obs_strategy/galaxy_counts_with_pixel_calibration.py` (`galaxy_counts_with_pixel_calibration`) - https://github.com/lsst/rubin_sim/blob/main/rubin_sim/maf/maf_contrib/lss_obs_strategy/galaxy_counts_with_pixel_calibration.py
  - `rubin_sim/maf/maf_contrib/lss_obs_strategy/constants_for_pipeline.py` (`power_law_const_a`, `power_law_const_b`, `normalization_constant`)
  - `rubin_sim/maf/metrics/exgal_m5.py` (`ExgalM5`), `rubin_sim/maf/metrics/simple_metrics.py` (`Coaddm5Metric`)
- Directory listing: https://github.com/lsst/rubin_sim/tree/main/rubin_sim/maf/maf_contrib/lss_obs_strategy
- summary.h5 / MAF outputs for standard runs: https://s3df.slac.stanford.edu/data/rubin/sim-data/sims_featureScheduler_runs5.3/maf/
- Table of simulations: https://usdf-maf.slac.stanford.edu/


## 1. Imports

In [ ]:
import os
import inspect
import warnings
from os.path import splitext, basename

import numpy as np
import pandas as pd
import healpy as hp
import matplotlib.pyplot as plt

import rubin_sim
import rubin_sim.maf as maf
import rubin_sim.maf.metrics as metrics
import rubin_sim.maf.slicers as slicers
import rubin_sim.maf.maps as maps
import rubin_sim.maf.metric_bundles as mb
from rubin_sim.maf.batches.common import standard_summary

from rubin_sim.maf.maf_contrib.lss_obs_strategy.galaxy_counts_metric_extended import (
    GalaxyCountsMetricExtended,
)
from rubin_sim.maf.maf_contrib.lss_obs_strategy.galaxy_counts_with_pixel_calibration import (
    galaxy_counts_with_pixel_calibration,
)
from rubin_sim.maf.maf_contrib.lss_obs_strategy.constants_for_pipeline import (
    power_law_const_a,
    power_law_const_b,
    normalization_constant,
)
from rubin_sim.maf.metrics.exgal_m5 import ExgalM5

print("rubin_sim version:", rubin_sim.__version__)
print("normalization_constant (mock catalog -> CFHTLS i<25.5):", normalization_constant)
print("Available redshift bins:", list(power_law_const_a.keys()) + ["all"])

## 2. Configuration

Same OpSim file and output-directory convention as the rest of the `06_MAF_DESC_TaskF` series (this notebook is filed as `07_galaxycounts.ipynb`, a companion piece rather than a renumbering of that series).

In [ ]:
opsim_fname = "/Users/dagoret/DATA/OpSim/sim_baseline/baseline_v5.3.6_10yrs.db"
assert os.path.isfile(opsim_fname), f"OpSim database not found: {opsim_fname}"

run_name = splitext(basename(opsim_fname))[0]
print("run_name:", run_name)

In [ ]:
NB_TAG = "GALCOUNTS"
data_dir = f"data_07_{NB_TAG}"
figs_dir = f"figs_07_{NB_TAG}"
os.makedirs(data_dir, exist_ok=True)
os.makedirs(figs_dir, exist_ok=True)
print("MAF output (data) directory :", os.path.abspath(data_dir))
print("Figures output directory    :", os.path.abspath(figs_dir))

In [ ]:
resultsDb = maf.db.ResultsDb(out_dir=data_dir)

## 3. The MAF classes/functions used for galaxy counts

In [ ]:
print(inspect.getdoc(GalaxyCountsMetricExtended))

In [ ]:
print(inspect.getdoc(galaxy_counts_with_pixel_calibration))

## 4. Configuration

Following the same WFD, non-DD footprint convention as notebooks 01-04 in the `06_MAF_DESC_TaskF` series. The `i` band is used throughout (the DESC LSS/3x2pt reference band); `GalaxyCountsMetricExtended`'s own `filter_col` selection would work on a multi-band `data_slice` too, but restricting the SQL query to `band='i'` up front is more efficient and matches the depth cuts used elsewhere in this notebook series.

In [ ]:
bandpass = "i"
nside = 128  # GalaxyCountsMetricExtended default; must match the slicer's nside (used for the sq-deg -> pixel scale factor)
sqlconstraint = f"band='{bandpass}' and scheduler_note not like 'DD%%'"
upper_mag_limit = 32.0  # GalaxyCountsMetricExtended default; the erfc completeness term truncates the practical integration range well before this

healpixslicer = slicers.HealpixSlicer(nside=nside, use_cache=False)
pix_area = hp.nside2pixarea(nside, degrees=True)
print(f"nside={nside} -> pixel area = {pix_area:.5f} deg^2, npix = {hp.nside2npix(nside)}")
print("sqlconstraint:", sqlconstraint)

## 5. Running `GalaxyCountsMetricExtended`: full-redshift-range galaxy counts

Baseline run: dust-extinction-corrected depth (`ExgalM5`), full redshift range (`redshift_bin='all'`), mock-catalog power laws normalized to CFHTLS `i<25.5` counts (`normalized_mock_catalog_counts=True`, `cfht_ls_counts=False`) - the standard configuration. A `SumMetric` summary gives the **total** galaxy count over the footprint (i.e. `sum(num_gal_per_pixel)`), alongside the usual `standard_summary()` per-pixel statistics.

In [ ]:
def galcount_summary():
    summary = [metrics.SumMetric(metric_name="Total Galaxy Counts")]
    summary.extend(standard_summary())
    return summary


galcount_metric = GalaxyCountsMetricExtended(
    nside=nside,
    filter_band=bandpass,
    upper_mag_limit=upper_mag_limit,
    include_dust_extinction=True,
    redshift_bin="all",
    cfht_ls_counts=False,
    normalized_mock_catalog_counts=True,
    metric_name="GalaxyCounts (all z, normalized mock catalog)",
)

galcount_bundle = mb.MetricBundle(
    galcount_metric,
    healpixslicer,
    sqlconstraint,
    run_name=run_name,
    summary_metrics=galcount_summary(),
)

bd = mb.make_bundles_dict_from_list([galcount_bundle])
bgroup = mb.MetricBundleGroup(bd, opsim_fname, out_dir=data_dir, results_db=resultsDb)
bgroup.run_all()

print(
    "Total galaxy count (normalized mock catalog, all z):",
    galcount_bundle.summary_values["Total Galaxy Counts"],
)
print("Median per-pixel galaxy count:", galcount_bundle.summary_values["Median"])

## 6. Sanity check: CFHTLS power law vs. normalized mock-catalog sum

Setting `cfht_ls_counts=True` (with `redshift_bin='all'`, as required) instead uses the CFHTLS power law directly (LSST Science Book eq. 3.7), bypassing the mock catalogs and their normalization entirely. Since `normalized_mock_catalog_counts=True` above was specifically calibrated to match this same CFHTLS relation at `i<25.5`, the two totals should agree reasonably well over the same footprint - this is a useful consistency check on the metric configuration, not an independent prediction.

In [ ]:
cfhtls_metric = GalaxyCountsMetricExtended(
    nside=nside,
    filter_band=bandpass,
    upper_mag_limit=upper_mag_limit,
    include_dust_extinction=True,
    redshift_bin="all",
    cfht_ls_counts=True,
    metric_name="GalaxyCounts (CFHTLS power law)",
)

cfhtls_bundle = mb.MetricBundle(
    cfhtls_metric,
    healpixslicer,
    sqlconstraint,
    run_name=run_name,
    summary_metrics=galcount_summary(),
)

bd_cfhtls = mb.make_bundles_dict_from_list([cfhtls_bundle])
bgroup_cfhtls = mb.MetricBundleGroup(bd_cfhtls, opsim_fname, out_dir=data_dir, results_db=resultsDb)
bgroup_cfhtls.run_all()

total_norm = galcount_bundle.summary_values["Total Galaxy Counts"]
total_cfhtls = cfhtls_bundle.summary_values["Total Galaxy Counts"]
print(f"Total galaxy count, normalized mock catalog (all z) : {total_norm:.4e}")
print(f"Total galaxy count, CFHTLS power law (i<{upper_mag_limit})  : {total_cfhtls:.4e}")
print(f"Ratio (mock/CFHTLS)                                  : {total_norm / total_cfhtls:.3f}")

## 7. Tomographic redshift bins

Running `GalaxyCountsMetricExtended` once per individual redshift bin (instead of `redshift_bin='all'`) gives the galaxy-count contribution of each DESC-style tomographic slice separately - directly relevant to the 3x2pt tomography discussed in notebook 01 (and in `02_MAF/science/DESC/01_sigma8tomography_demo.ipynb`), where the lens/source samples are themselves split into redshift bins.

In [ ]:
redshift_bins = list(power_law_const_a.keys())
print("Redshift bins:", redshift_bins)

zbin_bundles = {}
for zbin in redshift_bins:
    zbin_metric = GalaxyCountsMetricExtended(
        nside=nside,
        filter_band=bandpass,
        upper_mag_limit=upper_mag_limit,
        include_dust_extinction=True,
        redshift_bin=zbin,
        cfht_ls_counts=False,
        normalized_mock_catalog_counts=True,
        metric_name=f"GalaxyCounts z-bin {zbin}",
    )
    zbin_bundles[zbin] = mb.MetricBundle(
        zbin_metric,
        healpixslicer,
        sqlconstraint,
        run_name=run_name,
        summary_metrics=galcount_summary(),
    )

bd_zbins = mb.make_bundles_dict_from_list(list(zbin_bundles.values()))
bgroup_zbins = mb.MetricBundleGroup(bd_zbins, opsim_fname, out_dir=data_dir, results_db=resultsDb)
bgroup_zbins.run_all()
print("Done.")

In [ ]:
zbin_totals = pd.Series(
    {zbin: zbin_bundles[zbin].summary_values["Total Galaxy Counts"] for zbin in redshift_bins},
    name="Total Galaxy Counts",
).rename_axis("redshift_bin")

zbin_sum = zbin_totals.sum()
print(f"Sum over individual z-bins        : {zbin_sum:.4e}")
print(f"Direct 'all' redshift_bin total   : {total_norm:.4e}")
print(f"Ratio (sum of bins / 'all')       : {zbin_sum / total_norm:.3f}")

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(range(len(redshift_bins)), zbin_totals.values, color="steelblue")
ax.set_xticks(range(len(redshift_bins)))
ax.set_xticklabels(redshift_bins, rotation=45, ha="right")
ax.set_ylabel("Total galaxy counts")
ax.set_title(f"Galaxy counts per tomographic redshift bin ({bandpass}-band) - {run_name}")
ax.grid(alpha=0.3, axis="y")
fig.tight_layout()

base = os.path.join(figs_dir, f"{run_name}_galcounts_per_zbin")
fig.savefig(base + ".png", dpi=150, bbox_inches="tight")
fig.savefig(base + ".pdf", bbox_inches="tight")
print("Saved:", base + ".png/.pdf")
plt.show()

## 8. Healpix maps and histograms

In [ ]:
def save_bundle_plots(bundle, tag, figs_dir):
    made_plots = bundle.plot(savefig=False)
    saved = []
    for plot_type, fig in made_plots.items():
        if fig is None:
            continue
        base = os.path.join(figs_dir, f"{tag}_{plot_type}")
        fig.savefig(base + ".png", dpi=150, bbox_inches="tight")
        fig.savefig(base + ".pdf", bbox_inches="tight")
        saved.append(base)
        plt.close(fig)
    return saved

In [ ]:
print("--- Galaxy counts, all z (normalized mock catalog) ---")
saved = save_bundle_plots(galcount_bundle, f"{run_name}_GalaxyCounts_allz", figs_dir)
for s in saved:
    print("  saved:", s + ".png/.pdf")
_ = galcount_bundle.plot(savefig=False)
plt.show()

In [ ]:
example_zbin = redshift_bins[3]  # an intermediate-z example bin, e.g. '0.66<z<1.0'
print(f"--- Galaxy counts, z-bin {example_zbin} ---")
saved = save_bundle_plots(
    zbin_bundles[example_zbin], f"{run_name}_GalaxyCounts_zbin_{example_zbin}", figs_dir
)
for s in saved:
    print("  saved:", s + ".png/.pdf")
_ = zbin_bundles[example_zbin].plot(savefig=False)
plt.show()

## 9. Results table

In [ ]:
rows = [
    {"quantity": "All z, normalized mock catalog", "total_galaxy_counts": total_norm},
    {"quantity": f"All z, CFHTLS power law (i<{upper_mag_limit})", "total_galaxy_counts": total_cfhtls},
]
for zbin in redshift_bins:
    rows.append({"quantity": f"z-bin {zbin}", "total_galaxy_counts": zbin_totals[zbin]})

results_df = pd.DataFrame(rows).set_index("quantity")
results_df

In [ ]:
results_csv = os.path.join(data_dir, f"{run_name}_galaxycounts_summary.csv")
results_df.to_csv(results_csv)
print("Saved:", results_csv)

## 10. `galaxy_counts_with_pixel_calibration`: pixel-specific calibration-error demo

Unlike `GalaxyCountsMetricExtended`, this function takes a `coaddm5` value directly and does not go through a slicer - so it is the right tool to inject a per-pixel systematic that has nothing to do with the OpSim visit history itself, such as a **photometric calibration zero-point error map**. Workflow:

1. Compute a baseline `coaddm5` HEALpix map with the normal MAF chain (`ExgalM5`, dust-corrected, same `i`-band / non-DD footprint as above). A coarser `nside` is used here purely to keep the per-pixel Python loop (one `scipy.integrate.quad` call per pixel) fast for this illustrative demo - `GalaxyCountsMetricExtended` above uses the full `nside=128` because it is vectorized through the normal MAF bundle machinery instead.
2. Draw a toy per-pixel calibration-offset map: independent, zero-mean Gaussian offsets with `sigma_calib` mag scatter, representing an uncorrected photometric zero-point systematic (a simple stand-in for the "structure induced by the telescope observing strategy" systematics discussed in Awan et al. 2016; a real calibration-error map would instead come from a photometric-calibration pipeline).
3. Call `galaxy_counts_with_pixel_calibration` once per (unmasked) pixel, with and without the offset added to `coaddm5`, and compare the resulting galaxy-count maps.

In [ ]:
nside_demo = 16  # coarser than the main analysis, chosen only so the per-pixel Python loop below runs quickly
sqlconstraint_demo = sqlconstraint  # same i-band, non-DD footprint

coaddm5_metric = ExgalM5(m5_col="fiveSigmaDepth", metric_name="ExgalM5_demo")
healpixslicer_demo = slicers.HealpixSlicer(nside=nside_demo, use_cache=False)

coaddm5_bundle = mb.MetricBundle(
    coaddm5_metric,
    healpixslicer_demo,
    sqlconstraint_demo,
    run_name=run_name,
    summary_metrics=standard_summary(),
)
bd_coaddm5 = mb.make_bundles_dict_from_list([coaddm5_bundle])
bgroup_coaddm5 = mb.MetricBundleGroup(bd_coaddm5, opsim_fname, out_dir=data_dir, results_db=resultsDb)
bgroup_coaddm5.run_all()

coaddm5_map = coaddm5_bundle.metric_values  # masked array, mag, one value per pixel at nside_demo
print(f"nside_demo={nside_demo}, npix={hp.nside2npix(nside_demo)}, unmasked pixels={coaddm5_map.count()}")

In [ ]:
sigma_calib = 0.02  # mag, toy photometric calibration zero-point scatter (illustrative only)
rng = np.random.default_rng(seed=42)
calib_offset_map = rng.normal(loc=0.0, scale=sigma_calib, size=coaddm5_map.shape)

good_pix = np.where(~coaddm5_map.mask)[0]
print(
    f"Computing galaxy_counts_with_pixel_calibration for {len(good_pix)} unmasked pixels "
    f"(baseline + perturbed) ..."
)

num_gal_baseline = np.full(coaddm5_map.shape, hp.UNSEEN)
num_gal_perturbed = np.full(coaddm5_map.shape, hp.UNSEEN)

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    for ipix in good_pix:
        c5 = coaddm5_map[ipix]
        num_gal_baseline[ipix] = galaxy_counts_with_pixel_calibration(
            c5,
            upper_mag_limit=upper_mag_limit,
            nside=nside_demo,
            filter_band=bandpass,
            redshift_bin="all",
        )
        num_gal_perturbed[ipix] = galaxy_counts_with_pixel_calibration(
            c5 + calib_offset_map[ipix],
            upper_mag_limit=upper_mag_limit,
            nside=nside_demo,
            filter_band=bandpass,
            redshift_bin="all",
        )

num_gal_baseline = hp.ma(num_gal_baseline)
num_gal_perturbed = hp.ma(num_gal_perturbed)
num_gal_baseline.mask = coaddm5_map.mask
num_gal_perturbed.mask = coaddm5_map.mask

print("Done.")

In [ ]:
delta = num_gal_perturbed - num_gal_baseline
rel_delta = delta.compressed() / num_gal_baseline.compressed()

print(f"Baseline total galaxy count (nside={nside_demo})   : {num_gal_baseline.compressed().sum():.4e}")
print(
    f"Perturbed total galaxy count (calib sigma={sigma_calib} mag): {num_gal_perturbed.compressed().sum():.4e}"
)
print(f"Median |relative delta| per pixel                  : {np.median(np.abs(rel_delta)):.4f}")
print(f"95th percentile |relative delta| per pixel          : {np.percentile(np.abs(rel_delta), 95):.4f}")

In [ ]:
fig = plt.figure(figsize=(8, 5))
hp.mollview(
    delta.filled(hp.UNSEEN),
    fig=fig.number,
    title=f"Delta galaxy counts from a {sigma_calib} mag calibration-offset toy map - {run_name}",
    unit="galaxies/pixel",
    cmap="RdBu_r",
)
base = os.path.join(figs_dir, f"{run_name}_galcounts_calib_offset_map")
fig.savefig(base + ".png", dpi=150, bbox_inches="tight")
fig.savefig(base + ".pdf", bbox_inches="tight")
print("Saved:", base + ".png/.pdf")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
ax.hist(rel_delta, bins=40, color="steelblue", alpha=0.8)
ax.axvline(0.0, color="black", lw=1, ls="--")
ax.set_xlabel("Relative galaxy-count change per pixel\n(perturbed - baseline) / baseline")
ax.set_ylabel("Pixel count")
ax.set_title(f"Effect of a {sigma_calib} mag calibration-offset toy map on galaxy counts - {run_name}")
ax.grid(alpha=0.3)
fig.tight_layout()

base = os.path.join(figs_dir, f"{run_name}_galcounts_calib_offset_histogram")
fig.savefig(base + ".png", dpi=150, bbox_inches="tight")
fig.savefig(base + ".pdf", bbox_inches="tight")
print("Saved:", base + ".png/.pdf")
plt.show()

## 11. Caveats

- Both utilities model the galaxy luminosity function as mock-catalog power laws (Padilla et al. SAG semi-analytic model) per redshift bin, with only a simple color correction between bands and a single `erfc` completeness curve - there is no proper photometric selection function, no shape noise, no photo-z scatter, and no attempt to model any specific LSST/DESC redshift-estimation pipeline.
- `redshift_bin='all'` sums the ten individual power laws rather than re-fitting a single all-z power law, so the "all z" and "sum of z-bins" totals in Sections 5-7 are consistent by construction but both inherit the same underlying mock-catalog assumptions; neither should be read as a precision forecast of the actual LSST galaxy sample size.
- The `upper_mag_limit=32.0` default is far fainter than any realistic detection limit; the `erfc(m - coaddm5)` completeness term is what actually truncates the effective integration range near `coaddm5`, not `upper_mag_limit` itself.
- `nside` in `GalaxyCountsMetricExtended` and `galaxy_counts_with_pixel_calibration` only sets the sq-deg -> per-pixel **scale factor**; it must match the `HealpixSlicer`'s own `nside` (or, in Section 10, the coarser `nside_demo` actually used) or the returned per-pixel counts will be wrong by that scale factor.
- The Section 10 calibration-offset map is a simple, uncorrelated-Gaussian **toy model** for illustration only - it is not derived from any actual Rubin photometric-calibration error budget, and real calibration systematics are typically spatially correlated (e.g. tied to individual raft/CCD or tract boundaries) rather than independent per HEALpix pixel.
- The `i<25.5` and `sqlconstraint` (WFD, non-DD, `i` band) footprint choices here are arbitrary, matching the rest of this notebook series for comparability, not a specific DESC-approved LSS footprint definition.


## References
- Awan, H. et al. 2016, ApJ 829, 50, "Testing LSST Dither Strategies for Survey Uniformity and Large-Scale Structure Systematics" - defines the galaxy-count-from-depth model (mock-catalog power laws, incompleteness, calibration-error propagation) implemented by both utilities in this notebook. https://iopscience.iop.org/article/10.3847/0004-637X/829/1/50
- Lochner, M. et al. 2018, "Optimizing LSST Observing Strategy for Dark Energy Science", arXiv:1808.00006 - uses the `GalaxyCountsMetricExtended`-derived `depthLimitedNumGalMetric` to estimate galaxy counts feeding the same 3x2pt static-probes forecasts used in notebook 01.
- LSST Science Collaboration 2009, "LSST Science Book", arXiv:0912.0201 - source of the CFHTLS power law (eq. 3.7) used when `cfht_ls_counts=True`.
- `rubin_sim.maf` documentation: https://rubin-sim.lsst.io/maf.html
- `rubin_sim` source: https://github.com/lsst/rubin_sim (`rubin_sim/maf/maf_contrib/lss_obs_strategy/`)
